# Faz 1.8 — Katman 1: Açık Ağırlıklı LLM ile Rapor → Etiket

Bu notebook **iki modda** çalışır:

| Mod | Ne yapar | Süre |
|---|---|---|
| `MODE = "bakeoff"` | Aday modelleri 49 raporluk dev set'te yarıştırır, 20 gold'da per-label F1/AUC ölçer | ~30-60 dk |
| `MODE = "full"` | Kazanan modeli 4.407 raporun tamamına uygular | ölçülecek |

**Önce `bakeoff` çalıştır.** Hangi modelin yeterli olduğunu tahmin etmek yerine
ölçüyoruz: dev set'in 20'si gold etiketli, yani elle hiçbir şey işaretlemeden
anında yer gerçeği var.

## Üç tasarım kararı

**1. vLLM, Ollama değil.** Ollama etkileşimli tek-istek kullanımı için tasarlandı.
vLLM'in *continuous batching*'i onlarca isteği aynı anda GPU'da tutar — aynı
donanımda 10-30 kat throughput farkı. Ayrıca vLLM **JSON şema güdümlü decoding**
destekliyor: şemaya uymayan token'lar decode sırasında maskelenir, yani model
geçersiz JSON *üretemez*. 7B bir modelden 12 alanlı JSON isterken belirleyici.

**2. Güven skoru modele sorulmuyor.** Küçük modeller her şeye 0.9 der. Bunun
yerine `status` token'ının logprob'undan türetiliyor. İkisi de kaydediliyor;
hangisi doğrulukla daha iyi korele ediyorsa Katman 4'te o kullanılacak.

**3. `transformers` yedeği var.** Kaggle'da `pip install vllm` torch sürüm
çakışması yüzünden kırılabiliyor. Kırılırsa notebook `transformers` ile devam
eder — yavaş ama 49 rapor için yeterli, yani bake-off yine yapılır.

## Runtime ayarları

| Ayar | Değer |
|---|---|
| Accelerator | **GPU** (T4×2 varsa onu seç — 32 GB, büyük model için şart) |
| Internet | **Açık** (model indirilecek) |
| Persistence | No persistence |

## 0. vLLM — EN BAŞTA, torch import edilmeden önce

**Sıra burada kritik.** `pip install vllm` kendi torch sürümünü kurar. Eğer torch
bu hücreden önce import edilmişse, kurulum diskteki torch'u değiştirir ama
bellekte eski sürüm yüklü kalır → `import vllm` `ImportError` verir. Motor
çalışırken yağ değiştirmeye benziyor.

Bu yüzden kurulum **ilk kod hücresi** ve buradan önce hiçbir şey import
edilmiyor. Kurulum yeni yapıldıysa hücre sana kernel'i yeniden başlatmanı
söyleyecek — o zaman **Run → Restart & Run All**. İkinci turda vllm zaten
kurulu olacağı için kurulum atlanır ve import temiz çalışır.

In [ ]:
import importlib.util
import subprocess
import sys


def vllm_status():
    """('yok'|'hazir'|'kirik', bilgi) — hata MESAJINI da dondurur, sadece tipini degil."""
    if importlib.util.find_spec("vllm") is None:
        return "yok", None
    try:
        import vllm
        return "hazir", vllm.__version__
    except Exception as e:
        return "kirik", f"{type(e).__name__}: {e}"


TORCH_ALREADY_LOADED = "torch" in sys.modules
state, info = vllm_status()
print(f"vLLM durumu: {state}" + (f"  ({info})" if info else ""))
if TORCH_ALREADY_LOADED:
    print("  !! torch bu hucreden ONCE import edilmis — kurulum gerekirse")
    print("     kernel yeniden baslatilmali.")

if state == "yok":
    print()
    print("Kuruluyor (5-15 dk, torch'u da indirebilir)...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "vllm"],
                       capture_output=True, text=True, timeout=3600)
    if r.returncode != 0:
        print("  pip HATA (son 2000 karakter):")
        print((r.stderr or r.stdout)[-2000:])
    else:
        print("  pip tamam")
    state, info = vllm_status()
    print(f"vLLM durumu (kurulumdan sonra): {state}" + (f"  ({info})" if info else ""))

HAS_VLLM = state == "hazir"

if state == "kirik":
    print()
    print("!" * 72)
    print("vLLM kurulu ama import edilemiyor. TAM HATA:")
    print("  " + str(info))
    print()
    low = str(info).lower()
    if "torch" in low or "libc" in low or "symbol" in low or "cuda" in low:
        print("Bu bir SURUM UYUSMAZLIGI. Cozum:")
        print("  Run -> Restart & Run All   (kernel yeniden baslasin)")
        print("Yeniden baslattiktan sonra vllm zaten kurulu olacak, kurulum")
        print("atlanacak ve import temiz calisacak.")
    else:
        print("Surum uyusmazligina benzemiyor. Bu mesaji bana gonder —")
        print("kurulum satirini buna gore degistirecegiz.")
    print("!" * 72)
elif not HAS_VLLM:
    print()
    print("vLLM kurulamadi. transformers yoluna dusulecek:")
    print("  - 49 raporluk bake-off YAPILABILIR (yavas ama olur)")
    print("  - 4.407 raporluk tam kosu icin cok yavas — once vLLM cozulmeli")

## 0b. Güdümlü decoding sondası — model yüklemeden, 10 saniyede

İlk bake-off'ta güdümlü decoding **sessizce çalışmadı** ve bunu ancak 3 model
koştuktan ve ham çıktıları analiz ettikten sonra anladık. Bedeli: 4 rapor
tamamen kayboldu (%8), hepsi İngilizce olmayan.

Bu sonda model yüklemiyor, sadece `SamplingParams`'ın hangi alanı kabul
ettiğine bakıyor. **Buradaki cevap `HICBIRI CALISMIYOR` ise bake-off'u
çalıştırma** — çıktıyı gönder, sürüme göre düzeltelim.

In [ ]:
%%writefile /kaggle/working/vllm_probe.py
"""vLLM'in gudumlu decoding API'sini TESPIT ET — model yuklemeden, 10 saniyede.

Neden ayri bir sonda: ilk bake-off'ta gudumlu decoding sessizce calismadi ve
bunu ancak 3 model kostuktan ve ham ciktilari analiz ettikten sonra anladik.
Bu sonda model yuklemedigi icin saniyeler suruyor ve dogrudan cevap veriyor:
hangi API sekli bu vLLM surumunde kabul ediliyor?

Kaggle'da kullanim: vLLM kurulum hucresinden SONRA yeni bir hucreye yapistirip
calistir. Ciktiyi oldugu gibi gonder.
"""
import sys


def probe():
    print("=" * 68)
    print("vLLM GUDUMLU DECODING SONDASI")
    print("=" * 68)
    try:
        import vllm
    except Exception as e:
        print(f"vllm import EDILEMIYOR: {type(e).__name__}: {e}")
        return
    print("vllm surumu:", getattr(vllm, "__version__", "?"))
    print("python     :", sys.version.split()[0])

    try:
        import torch
        print("torch      :", torch.__version__)
    except Exception:
        pass

    # Hangi sinif adlari var?
    print()
    print("--- ilgili siniflar mevcut mu? ---")
    for modpath, name in [("vllm.sampling_params", "StructuredOutputsParams"),
                          ("vllm", "StructuredOutputsParams"),
                          ("vllm.sampling_params", "GuidedDecodingParams"),
                          ("vllm", "GuidedDecodingParams")]:
        try:
            mod = __import__(modpath, fromlist=[name])
            getattr(mod, name)
            print(f"  VAR  {modpath}.{name}")
        except Exception as e:
            print(f"  yok  {modpath}.{name}  ({type(e).__name__})")

    # SamplingParams hangi alanlari kabul ediyor?
    print()
    print("--- SamplingParams alanlari ---")
    from vllm import SamplingParams
    fields = None
    for attr in ("model_fields", "__dataclass_fields__", "__annotations__"):
        f = getattr(SamplingParams, attr, None)
        if f:
            fields = sorted(f.keys())
            print(f"  ({attr} uzerinden, {len(fields)} alan)")
            break
    if fields:
        ilgili = [f for f in fields
                  if any(s in f.lower() for s in
                         ("guid", "struct", "json", "schema", "logprob", "grammar"))]
        print("  ilgili alanlar:", ilgili or "(hicbiri)")
    else:
        print("  alan listesi alinamadi")

    # Gercekten insa edilebiliyor mu? (asil test)
    schema = {"type": "object",
              "properties": {"x": {"type": "string"}},
              "required": ["x"], "additionalProperties": False}
    base = dict(temperature=0.0, max_tokens=64, logprobs=5)

    print()
    print("--- INSA TESTI (asil cevap burada) ---")
    ok_any = False

    for modpath, clsname in [("vllm.sampling_params", "StructuredOutputsParams"),
                             ("vllm", "StructuredOutputsParams")]:
        try:
            mod = __import__(modpath, fromlist=[clsname])
            cls = getattr(mod, clsname)
            SamplingParams(structured_outputs=cls(json=schema), **base)
            print(f"  CALISIYOR  structured_outputs={clsname}(json=...)   [V1 API]")
            ok_any = True
            break
        except Exception as e:
            print(f"  olmadi     {clsname}: {type(e).__name__}: {str(e)[:110]}")

    try:
        from vllm.sampling_params import GuidedDecodingParams
        SamplingParams(guided_decoding=GuidedDecodingParams(json=schema), **base)
        print("  CALISIYOR  guided_decoding=GuidedDecodingParams(json=...)")
        ok_any = True
    except Exception as e:
        print(f"  olmadi     GuidedDecodingParams: {type(e).__name__}: {str(e)[:110]}")

    try:
        SamplingParams(guided_json=schema, **base)
        print("  CALISIYOR  guided_json=...")
        ok_any = True
    except Exception as e:
        print(f"  olmadi     guided_json: {type(e).__name__}: {str(e)[:110]}")

    print()
    print("=" * 68)
    if ok_any:
        print("SONUC: gudumlu decoding KULLANILABILIR -> bake-off'u calistir.")
    else:
        print("SONUC: HICBIRI CALISMIYOR. Yukaridaki hata mesajlarini gonder;")
        print("       vLLM surumune gore baska bir yol (orn. outlines/xgrammar")
        print("       backend'i veya surum sabitleme) gerekecek.")
    print("=" * 68)


if __name__ == "__main__":
    probe()

In [ ]:
if HAS_VLLM:
    # Kendi isim alaninda calistir: Jupyter hucresinde __name__ == "__main__"
    # oldugu icin exec, dosyanin sonundaki `if __name__ == "__main__": probe()`
    # satirini de tetikliyor ve cikti IKI KEZ basiliyordu.
    _ns = {"__name__": "vllm_probe"}
    exec(open("/kaggle/working/vllm_probe.py").read(), _ns)
    _ns["probe"]()
else:
    print("vLLM yok — sonda atlandi.")

## 1. Yapılandırma

In [ ]:
MODE = "retry"            # "retry" | "ablation" | "bakeoff" | "full"

# Aday modeller: kucukten buyuge. VRAM'e sigmayanlar otomatik atlanir.
# `vram_gb` = kabaca gereken bos VRAM (agirliklar + KV cache + aktivasyon).
# Bake-off 1 sonucuna gore guncellendi:
#  - fp16 modeller tp=2'ye alindi (tek 15 GB T4'e sigmiyorlar, OOM verdiler)
#  - 14B-AWQ tp=2'ye alindi: tp=1'de 32B'den YAVAS cikti (15.2 vs 10.6 s/rapor)
#  - Qwen3 modelleri kaldi ama artik enable_thinking=False ile calisacaklar
CANDIDATES = [
    {"id": "Qwen/Qwen2.5-14B-Instruct-AWQ", "vram_gb": 12, "tp": 2, "quant": "awq"},
    {"id": "Qwen/Qwen2.5-32B-Instruct-AWQ", "vram_gb": 24, "tp": 2, "quant": "awq"},
]

# --- ABLATION (Tur 3 sonucu) -------------------------------------------------
# 14B kazandi: F1_absent 0.6830 vs 32B 0.6805 (n=20, fark gurultu) ama
# 6.42 saat vs 12.41 — tek oturuma sigan tek model.
ABLATION_MODEL = {"id": "Qwen/Qwen2.5-14B-Instruct-AWQ", "vram_gb": 12,
                  "tp": 2, "quant": "awq"}

# Tur 1 -> Tur 3'te DORT sey birden degisti, o yuzden F1_absent dususunu
# (0.7251 -> 0.6805) prompt'a mi decoding'e mi atfedecegimizi bilmiyoruz.
# 14B artik hizli oldugu icin ayiklamak ~12 dakika. Tek degisken: 7. kural.
ABLATION_VARIANTS = [
    ("minimal_VAR", True),     # mevcut prompt
    ("minimal_YOK", False),    # "minimal/trace/mild -> uncertain" kurali cikarildi
]

# Tam kosuda kullanilacak model — bake-off sonucuna gore ELLE doldur.
# Bake-off kazanani: 14B, F1_absent 0.6830 (32B 0.6805) ve YARI surede.
FULL_RUN_MODEL = {"id": "Qwen/Qwen2.5-14B-Instruct-AWQ", "vram_gb": 12,
                  "tp": 2, "quant": "awq"}

# 4096 -> 8192. ONCEKI DEGER HATALIYDI ve olcum bunu acikca gosterdi:
#   prompt oneki  ~3.355 token (sistem 5.720 + few-shot 4.344 karakter)
#   4096 - 3355   =  ~740 token, rapor VE cikti icin TOPLAM
# Yani 1.000 karakterden uzun her rapor cikti butcesini 700'un altina itiyordu.
# Olculen sonuc (14B, Tur 3):
#   rapor  0-800 kar -> parse_ok 1.000      1200-1600 -> 0.727
#        800-1200    -> 0.867               2500-5000 -> 0.000
# Bu kesilme degil BUTCE SIKISMASI; vLLM girdi+cikti <= max_model_len olacak
# sekilde cikti butcesini kirpiyor ve JSON yarim kaliyor.
# 8192 ile: onek 3.355 + en uzun rapor ~1.581 + cikti 1.100 = ~6.036, rahat pay var.
MAX_MODEL_LEN = 8192
# 700 -> 1100: gudumlu decoding modeli 12 bulgunun HEPSINI doldurmaya zorluyor,
# kisa yoldan cikamiyor. Tur 3'te ortalama cikti 463-479 token'di ve parse_ok
# gudumlu decoding ACIKKEN dustu (0.918 -> 0.816/0.857) — kesilme suphesi.
# Kosu `n_capped` sayacini basiyor, hipotezi dogrudan test edecek.
# 1100 -> 1400: tam kosuda 53 rapor (%1.2) tavana dayanip kesildi ve
# ayristirilamadi (n_fail_capped == n_fail == 53, yani TEK sebep buydu).
# En uzun prompt 5.700 token, 5700+1400=7100 < 8192 — hala rahat.
MAX_NEW_TOKENS = 1400
OUT_DIR = "/kaggle/working"

# Token basina 5 aday logprob dondurmek buyuk bir serilestirme yuku ve guven
# olcumu zaten yapildi (kendi bildirdigi 0.819 > logprob 0.679). Yeniden olcmek
# istersen 5 yap — o zaman <etiket>_lpconf sutunlari dolar.
LOGPROBS = None

## 2. Modülleri diske yaz

Üç modül gömülü geliyor — repo'daki sürümleri tek doğru sürüm, notebook onları
`py2ipynb.py` dönüşümünde gömer, yani elle düzenlenmez ve bayatlamaz.

In [ ]:
%%writefile /kaggle/working/prompt.py
"""Katman 1 — rapor metninden 12 bulgunun yapilandirilmis cikarimi.

Saglayicidan BAGIMSIZ: burada sadece prompt metni, JSON semasi ve ayristirici var.
Hangi modele gonderilecegi cagiran tarafin isi (src/labels/run_extract.py).

Tasarim gerekcesi: ARCHITECTURE.md Bolum 2, Katman 1.
"""
import json
import re

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

# Etiket tanimlari prompt'a AYNEN giriyor. Bunlar yarismanin kendi tanimlari —
# ozellikle kompartman ayrimlari (Medial/Lateral/PF OA) modelin kafasini
# karistirabilecek tek yer, o yuzden acik yazildi.
LABEL_DEFINITIONS = """\
1.  ACL              — anterior cruciate ligament injury: tear (partial or complete),
                       sprain, rupture, or graft failure.
2.  MCL              — medial collateral ligament injury: tear, sprain, grade I-III.
3.  Medial Meniscus  — medial meniscus tear of any type (horizontal, vertical, radial,
                       complex, root tear, bucket-handle). Degeneration WITHOUT tear
                       does not count.
4.  Lateral Meniscus — same, for the lateral meniscus.
5.  Medial OA        — osteoarthritis of the MEDIAL tibiofemoral compartment:
                       cartilage loss, joint space narrowing, subchondral sclerosis,
                       osteophytes in that compartment.
6.  Lateral OA       — same, for the LATERAL tibiofemoral compartment.
7.  PF OA            — patellofemoral osteoarthritis: chondral loss / osteophytes
                       between patella and femoral trochlea. Note this is a SEPARATE
                       compartment from 5 and 6.
8.  Effusion         — joint effusion / intra-articular fluid accumulation.
9.  Synovitis        — synovial inflammation, synovial thickening, synovial enhancement.
                       Effusion ALONE is not synovitis.
10. Baker's          — Baker's cyst / popliteal cyst / parameniscal cyst in the
                       popliteal fossa.
11. Contusion        — bone contusion / bone marrow oedema / bone bruise WITHOUT a
                       fracture line.
12. Fracture         — any fracture: cortical, subchondral, avulsion, insufficiency,
                       stress fracture, osteochondral fracture with a fracture line."""

FEW_SHOT = [
    # Ingilizce — normal, kapsamli rapor. "absent" kararinin nasil verilecegini gosterir.
    {
        "report": "MRI right knee. Technique: sagittal, coronal and axial PD and "
                  "T2-weighted fat-saturated sequences. Findings: Cruciate and "
                  "collateral ligaments are intact. Both menisci are of normal "
                  "signal and morphology, no tear. Articular cartilage is preserved "
                  "in all three compartments. No joint effusion. No bone marrow "
                  "oedema or fracture. Impression: Normal MRI of the right knee.",
        "output": {
            "exam_completeness": "comprehensive",
            "findings": {
                "ACL": {"status": "absent", "confidence": 0.97, "evidence": "Cruciate ... ligaments are intact"},
                "MCL": {"status": "absent", "confidence": 0.96, "evidence": "collateral ligaments are intact"},
                "Medial Meniscus": {"status": "absent", "confidence": 0.97, "evidence": "Both menisci ... no tear"},
                "Lateral Meniscus": {"status": "absent", "confidence": 0.97, "evidence": "Both menisci ... no tear"},
                "Medial OA": {"status": "absent", "confidence": 0.93, "evidence": "cartilage is preserved in all three compartments"},
                "Lateral OA": {"status": "absent", "confidence": 0.93, "evidence": "cartilage is preserved in all three compartments"},
                "PF OA": {"status": "absent", "confidence": 0.93, "evidence": "cartilage is preserved in all three compartments"},
                "Effusion": {"status": "absent", "confidence": 0.97, "evidence": "No joint effusion"},
                "Synovitis": {"status": "absent", "confidence": 0.80, "evidence": ""},
                "Baker's": {"status": "absent", "confidence": 0.75, "evidence": ""},
                "Contusion": {"status": "absent", "confidence": 0.96, "evidence": "No bone marrow oedema"},
                "Fracture": {"status": "absent", "confidence": 0.97, "evidence": "or fracture"},
            },
        },
    },
    # Almanca — pozitif bulgular + kompartman ayrimi + negation ayni raporda.
    {
        "report": "MRT linkes Knie. Befund: Komplette Ruptur des vorderen Kreuzbandes. "
                  "Horizontalriss des Innenmeniskus im Hinterhorn. Aussenmeniskus "
                  "unauffaellig. Deutlicher Gelenkerguss. Retropatellar hoehergradige "
                  "Chondropathie mit Osteophyten. Medialer und lateraler "
                  "Femorotibialgelenkspalt regelrecht. Kein Knochenmarksoedem, keine "
                  "Fraktur. Kleine Bakerzyste.",
        "output": {
            "exam_completeness": "comprehensive",
            "findings": {
                "ACL": {"status": "present", "confidence": 0.98, "evidence": "Komplette Ruptur des vorderen Kreuzbandes"},
                "MCL": {"status": "uncertain", "confidence": 0.40, "evidence": ""},
                "Medial Meniscus": {"status": "present", "confidence": 0.96, "evidence": "Horizontalriss des Innenmeniskus"},
                "Lateral Meniscus": {"status": "absent", "confidence": 0.93, "evidence": "Aussenmeniskus unauffaellig"},
                "Medial OA": {"status": "absent", "confidence": 0.88, "evidence": "Medialer ... Femorotibialgelenkspalt regelrecht"},
                "Lateral OA": {"status": "absent", "confidence": 0.88, "evidence": "lateraler Femorotibialgelenkspalt regelrecht"},
                "PF OA": {"status": "present", "confidence": 0.92, "evidence": "Retropatellar hoehergradige Chondropathie mit Osteophyten"},
                "Effusion": {"status": "present", "confidence": 0.97, "evidence": "Deutlicher Gelenkerguss"},
                "Synovitis": {"status": "uncertain", "confidence": 0.30, "evidence": ""},
                "Baker's": {"status": "present", "confidence": 0.96, "evidence": "Kleine Bakerzyste"},
                "Contusion": {"status": "absent", "confidence": 0.95, "evidence": "Kein Knochenmarksoedem"},
                "Fracture": {"status": "absent", "confidence": 0.96, "evidence": "keine Fraktur"},
            },
        },
    },
    # Turkce + TELEGRAFIK rapor. Kisa raporda "absent" DEGIL "uncertain" denmesi
    # gerektigini ogretir — Faz 1A'da raporlarin %? kadari bu kadar kisa.
    {
        "report": "Diz MR. Bulgular: Medial menisküs posterior boynuzda yırtık. "
                  "Eklem içinde efüzyon mevcut.",
        "output": {
            "exam_completeness": "partial",
            "findings": {
                "ACL": {"status": "uncertain", "confidence": 0.20, "evidence": ""},
                "MCL": {"status": "uncertain", "confidence": 0.20, "evidence": ""},
                "Medial Meniscus": {"status": "present", "confidence": 0.96, "evidence": "Medial menisküs posterior boynuzda yırtık"},
                "Lateral Meniscus": {"status": "uncertain", "confidence": 0.25, "evidence": ""},
                "Medial OA": {"status": "uncertain", "confidence": 0.20, "evidence": ""},
                "Lateral OA": {"status": "uncertain", "confidence": 0.20, "evidence": ""},
                "PF OA": {"status": "uncertain", "confidence": 0.20, "evidence": ""},
                "Effusion": {"status": "present", "confidence": 0.96, "evidence": "Eklem içinde efüzyon mevcut"},
                "Synovitis": {"status": "uncertain", "confidence": 0.20, "evidence": ""},
                "Baker's": {"status": "uncertain", "confidence": 0.20, "evidence": ""},
                "Contusion": {"status": "uncertain", "confidence": 0.20, "evidence": ""},
                "Fracture": {"status": "uncertain", "confidence": 0.20, "evidence": ""},
            },
        },
    },
]

# "minimal -> uncertain" kurali AYRI tutuluyor cunku ABLATION adayi: Tur 3'te
# kapsam 0.788 -> 0.662 dustu ve F1_absent 0.7251 -> 0.6805 geriledi. Bas suphe
# bu kural, ama ayni turda gudumlu decoding de degistigi icin atif yapilamadi.
# Acip kapatarak olcebilmek icin sablondan soyuldu.
RULE_MINIMAL = '''7. MINIMAL FINDINGS. If the only wording is "minimal", "trace", "mild",
   "grade I", "questionable" or "suspected", prefer "uncertain" (0.4-0.6) over
   "present". A definite finding is what "present" is for.

'''

SYSTEM_PROMPT_TEMPLATE = """\
You extract structured findings from knee MRI radiology reports.

Reports come from 19+ imaging centres on 5 continents and are written in at least
9 different languages. Do not ask for or state the language — read it and work in it.

For EACH of the 12 findings below, output exactly one status:

  "present"   — the report states or strongly implies the finding IS there
  "absent"    — the report states the finding is NOT there, or states that the
                relevant structure is normal/intact
  "uncertain" — the report does not let you decide

THE 12 FINDINGS
{LABEL_DEFINITIONS}

RULES — read all of them, they are where mistakes happen.

1. NEGATION IS AN ANSWER, NOT A GAP. "No meniscal tear", "yırtık izlenmedi",
   "keine Fraktur", "sin derrame", "ligaments intact", "unauffällig",
   "regelrecht" → these mean "absent", NOT "uncertain". Getting negation right
   matters as much as getting positives right.

2. NOT MENTIONED depends on how complete the exam report is. First decide
   "exam_completeness":
     "comprehensive" — the report systematically walks through the knee
                       (ligaments, menisci, cartilage, fluid, bone) or ends with
                       a global normal statement ("Normal MRI of the knee")
     "partial"       — short, telegraphic, or clearly only reporting the
                       abnormality found
   Then, for a finding that is NOT mentioned at all:
     - comprehensive → "absent", confidence 0.70-0.90 (a thorough radiologist
       who saw it would have written it)
     - partial       → "uncertain", confidence 0.15-0.35 (silence tells you nothing)

3. COMPARTMENTS ARE SEPARATE. "Medial OA", "Lateral OA" and "PF OA" are three
   different findings. "Retropatellar chondropathy" is PF OA and says nothing
   about the tibiofemoral compartments. A generic "gonarthrosis" or
   "degenerative changes" without a named compartment → mark the compartments
   "uncertain" (0.4-0.5), do not spread it across all three as "present".

4. DISTINGUISH THESE PAIRS:
   - Effusion (fluid) vs Synovitis (inflamed synovium). Effusion alone is NOT
     synovitis.
   - Contusion (marrow oedema, no fracture line) vs Fracture (a fracture line).
     A report describing oedema AND a fracture line → both "present".

5. DEGENERATION AND OEDEMA ARE NOT INJURY. This applies to BOTH menisci AND
   ligaments, and it is the single most common mistake on these reports:
     - meniscus: "grade I/II degeneration", "intrasubstance signal", "linear
       signal without surfacing" → NOT a tear
     - ligament (ACL/MCL): "signal increase", "myxoid degeneration", "thinning",
       "periligamentous oedema", "fluid around the ligament", "laxity" → NOT a
       tear unless the report says tear / rupture / discontinuity
   Mark these "uncertain" (0.3-0.5), not "present". Only an explicit tear,
   rupture, discontinuity or named tear grade counts as "present".

6. NAMED SIDE BEATS THE HEADING. If the sentence says "lateral collateral
   ligament", that is NOT MCL, even in a report that mostly discusses the medial
   side. Read the structure actually named in the sentence you are quoting.

{RULE_MINIMAL}8. DO NOT INFER ACROSS FINDINGS. An ACL tear does not make an effusion
   "present". Judge each finding only on what the report says about it.

9. "confidence" is 0.0-1.0 and reflects how sure you are about the STATUS you
   chose, not how severe the finding is.

10. "evidence" is a SHORT verbatim quote from the report, in the report's own
   language, that justifies the status. Leave it "" when nothing in the text
   speaks to this finding. Never invent a quote.

OUTPUT
Return only a JSON object, no prose before or after:

{{"exam_completeness": "comprehensive" | "partial",
  "findings": {{"<finding name>": {{"status": "...", "confidence": 0.0, "evidence": "..."}}, ...}}}}

All 12 finding names must be present, spelled exactly as listed above."""


def build_system_prompt(with_minimal_rule: bool = True) -> str:
    """Sistem prompt'unu uret.

    `with_minimal_rule=False` 7. kurali ("minimal/trace/mild -> uncertain")
    tamamen cikarir. Kural numaralarinda bosluk olusur (6'dan 8'e atlar) ama bu
    onemsiz — modelin numaralara degil iceriklerine bakmasi gerekiyor ve
    ablationi kural metinlerini yeniden numaralandirmadan yapmak, iki varyant
    arasindaki TEK farkin o kural olmasini garanti ediyor.
    """
    return SYSTEM_PROMPT_TEMPLATE.format(
        LABEL_DEFINITIONS=LABEL_DEFINITIONS,
        RULE_MINIMAL=RULE_MINIMAL if with_minimal_rule else "")


SYSTEM_PROMPT = build_system_prompt(True)


def build_user_message(report: str) -> str:
    return f"REPORT:\n{report.strip()}\n\nExtract the 12 findings as JSON."


def build_few_shot_messages() -> list:
    """Few-shot ornekleri messages dizisine cevir (user/assistant ciftleri)."""
    msgs = []
    for ex in FEW_SHOT:
        msgs.append({"role": "user", "content": build_user_message(ex["report"])})
        msgs.append({"role": "assistant",
                     "content": json.dumps(ex["output"], ensure_ascii=False)})
    return msgs


# --- JSON semasi: structured output destekleyen saglayicilar icin ------------
def output_schema() -> dict:
    finding = {
        "type": "object",
        "properties": {
            "status": {"type": "string", "enum": ["present", "absent", "uncertain"]},
            "confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
            "evidence": {"type": "string"},
        },
        "required": ["status", "confidence", "evidence"],
        "additionalProperties": False,
    }
    return {
        "type": "object",
        "properties": {
            "exam_completeness": {"type": "string",
                                  "enum": ["comprehensive", "partial"]},
            "findings": {
                "type": "object",
                "properties": {k: finding for k in LABELS},
                "required": list(LABELS),
                "additionalProperties": False,
            },
        },
        "required": ["exam_completeness", "findings"],
        "additionalProperties": False,
    }


# --- Ayristirma -------------------------------------------------------------
_STATUS_TO_SOFT = {"present": 1.0, "absent": 0.0, "uncertain": None}


def _extract_json(text: str):
    """Modelin cevabindan JSON nesnesini cikar.

    Structured output kullanildiginda gereksiz, ama her saglayici desteklemiyor
    ve model bazen ```json blogu veya aciklama metni ekliyor. Once duz parse,
    sonra kod blogu, sonra ilk dengeli suslu parantez blogu.
    """
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"```(?:json)?\s*(.+?)\s*```", text, re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i, ch in enumerate(text[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except Exception:
                    return None
    return None


def parse_response(text: str, study_uid: str = None) -> dict:
    """Model cevabini duz bir sozluge cevir.

    Dondurulen sozlukte her etiket icin uc alan olur:
      <etiket>            soft hedef (1.0 / 0.0 / None)  -> egitim hedefi
      <etiket>_status     present/absent/uncertain/missing
      <etiket>_conf       0-1
      <etiket>_evidence   dayanak alinti
    Ayrica: parse_ok, exam_completeness, n_missing, raw_error
    """
    out = {"StudyInstanceUID": study_uid, "parse_ok": False,
           "exam_completeness": None, "n_missing": len(LABELS), "raw_error": None}
    data = _extract_json(text)
    if not isinstance(data, dict):
        out["raw_error"] = "JSON cikarilamadi"
        for k in LABELS:
            out[k], out[f"{k}_status"] = None, "missing"
            out[f"{k}_conf"], out[f"{k}_evidence"] = 0.0, ""
        return out

    out["exam_completeness"] = data.get("exam_completeness")
    findings = data.get("findings") or {}
    missing = 0
    for k in LABELS:
        f = findings.get(k)
        if not isinstance(f, dict):
            out[k], out[f"{k}_status"] = None, "missing"
            out[f"{k}_conf"], out[f"{k}_evidence"] = 0.0, ""
            missing += 1
            continue
        st = str(f.get("status", "")).strip().lower()
        if st not in _STATUS_TO_SOFT:
            st = "uncertain"
        try:
            conf = float(f.get("confidence", 0.0))
        except Exception:
            conf = 0.0
        out[k] = _STATUS_TO_SOFT[st]
        out[f"{k}_status"] = st
        out[f"{k}_conf"] = min(max(conf, 0.0), 1.0)
        out[f"{k}_evidence"] = str(f.get("evidence", ""))[:300]
    out["n_missing"] = missing
    out["parse_ok"] = missing == 0
    return out


# --- Soft label uretimi -----------------------------------------------------
# OLCULDU (Faz 1.8, Qwen2.5-32B-AWQ, 20 gold study):
#     sert 0/1 (uncertain=0.5)   macro AUC = 0.8323
#     0.5 +/- 0.5*conf           macro AUC = 0.8655   <- +0.033, bedava
#
# Mekanizma: sert etiketlerde butun "present"ler 1.0'da esitlenir ve AUC
# esitlikleri 0.5 sayar. Guven skoru o esitlikleri kirar; guven dogrulukla
# korele oldugu icin (AUC_guven 0.819) kirilma dogru yone gider.
#
# Bu, over-calling'i prompt'la kovalamak yerine SIRALAMAYA birakmak demek —
# yarismanin metrigi zaten macro ROC-AUC, mutlak kalibrasyon degil.
SOFT_UNCERTAIN = 0.5


def soft_label(status, confidence, uncertain_value: float = SOFT_UNCERTAIN):
    """(status, confidence) -> [0,1] araliginda soft hedef.

    present  -> 0.5 + 0.5*conf   (guven arttikca 1'e yaklasir)
    absent   -> 0.5 - 0.5*conf   (guven arttikca 0'a yaklasir)
    digeri   -> uncertain_value  (None verilirse loss'ta maskelenir)

    Boylece "present ama emin degil" ile "present ve emin" ayrilir; sert
    esiklemede ikisi de 1.0 olurdu.
    """
    try:
        c = float(confidence)
    except (TypeError, ValueError):
        c = 0.0
    c = min(max(c, 0.0), 1.0)
    if status == "present":
        return 0.5 + 0.5 * c
    if status == "absent":
        return 0.5 - 0.5 * c
    return uncertain_value


def add_soft_labels(df, conf_suffix: str = "_conf", uncertain_value: float = SOFT_UNCERTAIN):
    """Her etiket icin `<etiket>_soft` sutunu ekle (df yerinde degistirilir).

    `conf_suffix` neden parametre: bake-off'ta modelin kendi bildirdigi guven
    (`_conf`, AUC_guven 0.819) logprob'dan turetilenden (`_lpconf`, 0.679) daha
    iyi cikti, ama bu olcum modele bagli — degistirilebilir olsun.
    """
    for k in LABELS:
        st = df.get(f"{k}_status")
        cf = df.get(f"{k}{conf_suffix}")
        if st is None:
            continue
        if cf is None:
            cf = [None] * len(df)
        df[f"{k}_soft"] = [soft_label(s, c, uncertain_value) for s, c in zip(st, cf)]
    return df

In [ ]:
%%writefile /kaggle/working/evaluate.py
"""Katman 3 — weak-labeler'i gold uzerinde degerlendir.

Merkezi tasarim sorusu: `uncertain` ciktisini nasil sayalim?

Uc yol var ve UCU DE raporlanmali, cunku farkli sorulara cevap veriyorlar:

  maskele  : uncertain olanlari metrikten CIKAR
             -> "karar verdiginde ne kadar dogru?" (precision/recall karar verilen
                alt kumede). Kapsami gizler: her seye uncertain diyen bir model
                burada mukemmel gorunur.
  absent   : uncertain -> 0 kabul et
             -> gercekci senaryo, cunku egitimde uncertain'i maskelemezsek boyle
                davranmis oluruz. Recall'u dusurur.
  0.5      : uncertain -> 0.5 soft hedef
             -> AUC icin dogru yol: siralamaya katilir ama ne pozitif ne negatif
                tarafa tam agirlik verir.

AUC ayrica ozel: yarismanin metrigi macro ROC-AUC, yani weak-labeler'in
SIRALAMA kalitesi F1'inden daha alakali. Bu yuzden hem F1 hem AUC hesaplaniyor.
"""
import numpy as np
import pandas as pd

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]


def _auc(y, s):
    """ROC-AUC, sklearn'e bagimlilik olmadan (Mann-Whitney U, baglar 0.5 sayilir)."""
    y = np.asarray(y, float)
    s = np.asarray(s, float)
    m = np.isfinite(y) & np.isfinite(s)
    y, s = y[m], s[m]
    n1, n0 = int((y == 1).sum()), int((y == 0).sum())
    if n1 == 0 or n0 == 0:
        return np.nan
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s), float)
    ranks[order] = np.arange(1, len(s) + 1)
    # bagli skorlara ortalama rank ver
    df = pd.DataFrame({"s": s, "r": ranks})
    ranks = df.groupby("s")["r"].transform("mean").to_numpy()
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else np.nan
    r = tp / (tp + fn) if (tp + fn) else np.nan
    f = 2 * p * r / (p + r) if (p and r and (p + r) > 0) else (0.0 if (tp + fp + fn) else np.nan)
    return p, r, f


def evaluate(pred: pd.DataFrame, gold: pd.DataFrame, uncertain="mask") -> pd.DataFrame:
    """Per-label metrikler.

    pred : StudyInstanceUID + her etiket icin <k> (1/0/None) ve <k>_status
    gold : StudyInstanceUID + her etiket icin 0/1
    uncertain: "mask" | "absent" | "half"
    """
    g = gold.set_index("StudyInstanceUID")
    p = pred.set_index("StudyInstanceUID")
    common = g.index.intersection(p.index)
    g, p = g.loc[common], p.loc[common]

    rows = []
    for k in LABELS:
        if k not in g.columns or k not in p.columns:
            continue
        yt = g[k].astype(float).to_numpy()
        raw = p[k].to_numpy(dtype=object)
        st = p.get(f"{k}_status", pd.Series(index=p.index, dtype=object)).to_numpy()

        is_unc = np.array([s in ("uncertain", "missing") or v is None
                           for s, v in zip(st, raw)])
        yp = np.array([np.nan if v is None else float(v) for v in raw])

        if uncertain == "mask":
            keep = ~is_unc & np.isfinite(yt)
        elif uncertain == "absent":
            yp = np.where(is_unc, 0.0, yp)
            keep = np.isfinite(yt)
        elif uncertain == "half":
            yp = np.where(is_unc, 0.5, yp)
            keep = np.isfinite(yt)
        else:
            raise ValueError(uncertain)

        yt_k, yp_k = yt[keep], yp[keep]
        hard = (yp_k >= 0.5).astype(int)
        tp = int(((hard == 1) & (yt_k == 1)).sum())
        fp = int(((hard == 1) & (yt_k == 0)).sum())
        fn = int(((hard == 0) & (yt_k == 1)).sum())
        tn = int(((hard == 0) & (yt_k == 0)).sum())
        prec, rec, f1 = _prf(tp, fp, fn)

        rows.append({
            "etiket": k,
            "n_deger": int(keep.sum()),
            "kapsam": round(float((~is_unc).mean()), 3),
            "n_gold_poz": int((yt == 1).sum()),
            "precision": round(prec, 3) if np.isfinite(prec) else np.nan,
            "recall": round(rec, 3) if np.isfinite(rec) else np.nan,
            "F1": round(f1, 3) if np.isfinite(f1) else np.nan,
            "AUC": round(_auc(yt_k, yp_k), 3),
            "dogruluk": round((tp + tn) / max(len(yt_k), 1), 3),
        })
    return pd.DataFrame(rows)


def summarise(ev: pd.DataFrame) -> dict:
    return {
        "macro_F1": round(float(ev["F1"].mean(skipna=True)), 4),
        "macro_AUC": round(float(ev["AUC"].mean(skipna=True)), 4),
        "min_F1": round(float(ev["F1"].min(skipna=True)), 3),
        "en_zayif": ev.loc[ev["F1"].idxmin(), "etiket"] if ev["F1"].notna().any() else None,
        "ort_kapsam": round(float(ev["kapsam"].mean()), 3),
    }


def confidence_quality(pred: pd.DataFrame, gold: pd.DataFrame,
                       conf_suffix="_conf") -> pd.DataFrame:
    """Guven skoru gercekten dogrulukla korele mi?

    Katman 4 weak-label'lari guvene gore agirliklandiracak. Guven rastgeleyse
    agirliklandirma zarar verir. Bunu iki secenek icin ayri ayri olculuyoruz:
      _conf   : modelin kendi bildirdigi guven
      _lpconf : status token'inin logprob'undan turetilen guven
    """
    g = gold.set_index("StudyInstanceUID")
    p = pred.set_index("StudyInstanceUID")
    common = g.index.intersection(p.index)
    g, p = g.loc[common], p.loc[common]

    rows = []
    for k in LABELS:
        col = f"{k}{conf_suffix}"
        if k not in g.columns or col not in p.columns:
            continue
        yt = g[k].astype(float).to_numpy()
        raw = p[k].to_numpy(dtype=object)
        conf = pd.to_numeric(p[col], errors="coerce").to_numpy()
        m = np.array([v is not None for v in raw]) & np.isfinite(conf) & np.isfinite(yt)
        if m.sum() < 8:
            continue
        correct = np.array([float(v) == t for v, t in zip(raw[m], yt[m])], float)
        c = conf[m]
        if len(np.unique(c)) < 2 or len(np.unique(correct)) < 2:
            rows.append({"etiket": k, "n": int(m.sum()), "AUC_guven": np.nan,
                         "dogru_ort": np.nan, "yanlis_ort": np.nan})
            continue
        rows.append({
            "etiket": k, "n": int(m.sum()),
            "AUC_guven": round(_auc(correct, c), 3),
            "dogru_ort": round(float(c[correct == 1].mean()), 3),
            "yanlis_ort": round(float(c[correct == 0].mean()), 3),
        })
    return pd.DataFrame(rows)

In [ ]:
%%writefile /kaggle/working/devset_ids.py
"""tools/make_devset.py tarafindan URETILDI — elle duzenlemeyin.

Faz 1.8 dev set'i ve gold ayrimi. UID listeleri koda gomulu ki
notebooks/03_weak_labels_llm Kaggle'da ek dataset olmadan calissin.
"""

DEVSET = [
    "1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312",
    "1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274",
    "1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238",
    "1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187",
    "1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192",
    "1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973",
    "1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378",
    "1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465",
    "1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935",
    "1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828",
    "1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363",
    "1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903",
    "1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608",
    "1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039",
    "1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220",
    "1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911",
    "1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803",
    "1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003",
    "1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995",
    "1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500",
    "1.2.826.0.1.3680043.8.498.91168270990167204047014161851340054645",
    "1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152",
    "1.2.826.0.1.3680043.8.498.10299618611626190927173051117228647808",
    "1.2.826.0.1.3680043.8.498.10351460911387106879135924409514835385",
    "1.2.826.0.1.3680043.8.498.12864336642218457767597053831458716404",
    "1.2.826.0.1.3680043.8.498.13242536312535334908597493894251783308",
    "1.2.826.0.1.3680043.8.498.53370670740736715669115043403494610200",
    "1.2.826.0.1.3680043.8.498.13430522388784978790587514142693914965",
    "1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928",
    "1.2.826.0.1.3680043.8.498.54099199514076260519579096546600505763",
    "1.2.826.0.1.3680043.8.498.27671779587445983390840223475173360482",
    "1.2.826.0.1.3680043.8.498.35026052487843947230089350615730580521",
    "1.2.826.0.1.3680043.8.498.80290749690597272632725356393056746299",
    "1.2.826.0.1.3680043.8.498.11338962847958062668991874160027887372",
    "1.2.826.0.1.3680043.8.498.15024602758408626305833210011882949904",
    "1.2.826.0.1.3680043.8.498.96425049974772570897047688166336162674",
    "1.2.826.0.1.3680043.8.498.35790589718590759254855277551289842944",
    "1.2.826.0.1.3680043.8.498.11954711639198972958656966617027374769",
    "1.2.826.0.1.3680043.8.498.91940000162714863613139612302783091395",
    "1.2.826.0.1.3680043.8.498.13151845892175420505816340845155354956",
    "1.2.826.0.1.3680043.8.498.94955017957653373798729896600270551224",
    "1.2.826.0.1.3680043.8.498.33114390259375873203134285352951765534",
    "1.2.826.0.1.3680043.8.498.11722185671504770687850569082348083612",
    "1.2.826.0.1.3680043.8.498.15219410096632334470101257478715035071",
    "1.2.826.0.1.3680043.8.498.11396451979734570993257791565721573849",
    "1.2.826.0.1.3680043.8.498.88896246957673838955012057210098034177",
    "1.2.826.0.1.3680043.8.498.95588576819758266413765165887081039863",
    "1.2.826.0.1.3680043.8.498.41852534027066121760361035269460584004",
    "1.2.826.0.1.3680043.8.498.94002633501154835018549300667479350401",
]

GOLD_DEV = [
    "1.2.826.0.1.3680043.8.498.94433753471890306108089557761231739312",
    "1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274",
    "1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238",
    "1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187",
    "1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192",
    "1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973",
    "1.2.826.0.1.3680043.8.498.18392509497170616983977319528036573378",
    "1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465",
    "1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935",
    "1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828",
    "1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363",
    "1.2.826.0.1.3680043.8.498.13335129881737731410081002627729200903",
    "1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608",
    "1.2.826.0.1.3680043.8.498.64703506772167798469048460791472465039",
    "1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220",
    "1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911",
    "1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803",
    "1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003",
    "1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995",
    "1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500",
]

GOLD_HOLDOUT = [
    "1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149",
    "1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269",
    "1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052",
    "1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580",
    "1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840",
    "1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756",
    "1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604",
    "1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581",
    "1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945",
    "1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819",
    "1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492",
    "1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891",
    "1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309",
    "1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711",
    "1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404",
    "1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645",
    "1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924",
    "1.2.826.0.1.3680043.8.498.29764868091238287072166823522853419550",
    "1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945",
    "1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742",
    "1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683",
    "1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125",
    "1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132",
    "1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450",
    "1.2.826.0.1.3680043.8.498.59483999067816153759785789654766826710",
    "1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395",
    "1.2.826.0.1.3680043.8.498.62549354677638403845904556149868367236",
    "1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910",
    "1.2.826.0.1.3680043.8.498.65708905118633339771181857781063784327",
    "1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774",
    "1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002",
    "1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921",
    "1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190",
    "1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254",
    "1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037",
    "1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435",
    "1.2.826.0.1.3680043.8.498.90283565381042081768587894596970552767",
    "1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662",
]

assert not (set(GOLD_DEV) & set(GOLD_HOLDOUT)), "gold_dev ve gold_holdout kesisiyor"

In [ ]:
%%writefile /kaggle/working/retry_ids.py
"""tools/ ile URETILDI — Faz 1.8 tam kosusunda kesilen raporlar.

53 rapor (%1.2) 1100 token tavanina dayanip
kesildi; JSON yarim kaldi ve ayristirilamadi. Tamaminin sebebi ayni
(n_fail_capped == n_fail), yani max_tokens=1400 ile duzelmeli.
Sadece bunlari yeniden islemek 4.407 raporu bastan almaktan ~80 kat hizli.
"""

RETRY = [
    "1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156",
    "1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593",
    "1.2.826.0.1.3680043.8.498.10761658878615069270304502752374858610",
    "1.2.826.0.1.3680043.8.498.10975347305662629858093628672419901528",
    "1.2.826.0.1.3680043.8.498.11477583622065195478886525904745372063",
    "1.2.826.0.1.3680043.8.498.11915933561045570822185908784072057715",
    "1.2.826.0.1.3680043.8.498.11927352926669104658144933891780551483",
    "1.2.826.0.1.3680043.8.498.12005327370242618323737652892230673473",
    "1.2.826.0.1.3680043.8.498.12104799278874963688461235330096856415",
    "1.2.826.0.1.3680043.8.498.12619895582462244920124420363095282842",
    "1.2.826.0.1.3680043.8.498.13116941594523500503401720636978356127",
    "1.2.826.0.1.3680043.8.498.13363562142644508091380459785810556567",
    "1.2.826.0.1.3680043.8.498.14479913217228604289584726096114526268",
    "1.2.826.0.1.3680043.8.498.17070139245390396445395414066427036835",
    "1.2.826.0.1.3680043.8.498.17274490471295558668613733667390457509",
    "1.2.826.0.1.3680043.8.498.19374331384306309057311171252852774841",
    "1.2.826.0.1.3680043.8.498.23394601328148358087542766577727876635",
    "1.2.826.0.1.3680043.8.498.24555528844026121007453886375651403314",
    "1.2.826.0.1.3680043.8.498.24563801799946310644914912532825629118",
    "1.2.826.0.1.3680043.8.498.24661137747502154216858169258339731263",
    "1.2.826.0.1.3680043.8.498.26351577824204863242390892891061594387",
    "1.2.826.0.1.3680043.8.498.28946722216615442736212359468410891872",
    "1.2.826.0.1.3680043.8.498.38333934623775971943734069301776225018",
    "1.2.826.0.1.3680043.8.498.40930045208346676628855922755791787949",
    "1.2.826.0.1.3680043.8.498.42829490507052740062127365747846030214",
    "1.2.826.0.1.3680043.8.498.44879493336213912007673875382158420811",
    "1.2.826.0.1.3680043.8.498.46001596547902477402837300565271742663",
    "1.2.826.0.1.3680043.8.498.46407639440476738746130651250276012493",
    "1.2.826.0.1.3680043.8.498.47666607805434428477774854287399691400",
    "1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710",
    "1.2.826.0.1.3680043.8.498.49441835811239898312243678447796777305",
    "1.2.826.0.1.3680043.8.498.50689256431939553935971505794681061679",
    "1.2.826.0.1.3680043.8.498.51981019698056120061499970572334688555",
    "1.2.826.0.1.3680043.8.498.59948559729295284519115137429127740378",
    "1.2.826.0.1.3680043.8.498.60633008040453481523409852849000823589",
    "1.2.826.0.1.3680043.8.498.61148578767041261085865406067025346106",
    "1.2.826.0.1.3680043.8.498.63151733210715530368323422018984705316",
    "1.2.826.0.1.3680043.8.498.64752580249608101443758975861213065048",
    "1.2.826.0.1.3680043.8.498.66296018797614563698471347079960740755",
    "1.2.826.0.1.3680043.8.498.66401771232064170830621215861092532421",
    "1.2.826.0.1.3680043.8.498.67690090498828405347380696553164435363",
    "1.2.826.0.1.3680043.8.498.69033651423155214548081618535669641812",
    "1.2.826.0.1.3680043.8.498.73869775636181013203432845292462040922",
    "1.2.826.0.1.3680043.8.498.74177736542522869266952610634091628727",
    "1.2.826.0.1.3680043.8.498.77801860555102174958520570751272642387",
    "1.2.826.0.1.3680043.8.498.83152040259754415849193046299744699239",
    "1.2.826.0.1.3680043.8.498.83456649088980869429062498344457162631",
    "1.2.826.0.1.3680043.8.498.85333821516157983299252782678788406019",
    "1.2.826.0.1.3680043.8.498.85566900782101872452970957532250214590",
    "1.2.826.0.1.3680043.8.498.85883562680953255833567595509408888410",
    "1.2.826.0.1.3680043.8.498.92301915122994899889226415477164044164",
    "1.2.826.0.1.3680043.8.498.93062749017302143105207135454940053142",
    "1.2.826.0.1.3680043.8.498.93359387652821116106937882388838959209",
]

In [ ]:
%%writefile /kaggle/working/vllm_runner.py
"""Katman 1 — acik agirlikli LLM ile rapor cikarimi (vLLM).

Neden vLLM, neden Ollama degil:
  Ollama etkilesimli tek-istek kullanimi icin tasarlandi. vLLM'in continuous
  batching'i onlarca istegi ayni anda GPU'da tutar — ayni donanimda 10-30 kat
  throughput farki. Ayrica vLLM JSON sema gudumlu decoding destekliyor: semaya
  uymayan token'lar decode sirasinda maskelenir, yani model GECERSIZ JSON
  URETEMEZ. 7B bir modelden 12 alanli JSON isterken bu belirleyici.

Tasarim notlari:
  * Guven skoru MODELE SORULMUYOR (kucuk modeller her seye 0.9 der); status
    token'inin logprob'undan turetiliyor. Ikisi de kaydediliyor, hangisinin
    dogrulukla daha iyi korele ettigi dev set'te olculecek (Katman 4 girdisi).
  * Her cevap aninda JSONL'e yazilir -> Kaggle oturumu kesilirse kaldigi yerden
    devam eder. 4.400 raporluk bir kosuyu bastan almak istemiyoruz.
  * vLLM'in gudumlu decoding API'si surumler arasi degisti; uc bilinen sekli
    sirayla denenip calisan kullanilir, hicbiri yoksa gudumsuz moda dusulur
    (ayristiricimiz zaten dayanikli).
"""
import json
import math
import os
import sys
import time

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import prompt as P

STATUS_WORDS = ("present", "absent", "uncertain")


# --------------------------------------------------------------------------
# Gudumlu decoding: surum tespiti
# --------------------------------------------------------------------------
def make_sampling_params(schema, max_tokens=700, temperature=0.0, logprobs=None):
    """(SamplingParams, kullanilan_yol) dondur.

    vLLM surum farklari (ilk bake-off'ta hepsi basarisiz oldu -> gudumsuz kaldi,
    parse_ok 0.918'de takildi ve 4 rapor tamamen kayboldu):
      V1 engine   SamplingParams(structured_outputs=StructuredOutputsParams(json=...))
      >=0.6.x     SamplingParams(guided_decoding=GuidedDecodingParams(json=...))
      ~0.5.x      SamplingParams(..., guided_json=schema)
      eski        yok -> gudumsuz + dayanikli ayristirici

    Kullanicinin logunda "EngineCore" satirlari vardi, yani V1 engine calisiyor;
    o surumde alan adi `structured_outputs` olarak degisti. Ilk siraya o eklendi.
    """
    from vllm import SamplingParams

    base = dict(temperature=temperature, max_tokens=max_tokens, logprobs=logprobs)
    attempts = []

    # --- 1) vLLM V1: structured_outputs ---
    for modpath, clsname in [("vllm.sampling_params", "StructuredOutputsParams"),
                             ("vllm", "StructuredOutputsParams")]:
        try:
            mod = __import__(modpath, fromlist=[clsname])
            cls = getattr(mod, clsname)
            sp = SamplingParams(structured_outputs=cls(json=schema), **base)
            return sp, f"structured_outputs={clsname}(json=...)  [V1]"
        except Exception as e:
            attempts.append(f"{clsname}: {type(e).__name__}")

    # --- 2) guided_decoding=GuidedDecodingParams ---
    try:
        from vllm.sampling_params import GuidedDecodingParams
        sp = SamplingParams(guided_decoding=GuidedDecodingParams(json=schema), **base)
        return sp, "guided_decoding=GuidedDecodingParams(json=...)"
    except Exception as e:
        attempts.append(f"GuidedDecodingParams: {type(e).__name__}")

    # --- 3) duz guided_json ---
    try:
        sp = SamplingParams(guided_json=schema, **base)
        return sp, "guided_json=..."
    except Exception as e:
        attempts.append(f"guided_json: {type(e).__name__}")

    print("  [UYARI] GUDUMLU DECODING YOK -> gudumsuz mod.")
    print("          Denenen yollar: " + " | ".join(attempts))
    print("          Ayristirma hatasi orani yukselecek (ilk kosuda %8 idi).")
    return SamplingParams(**base), "GUDUMSUZ"


# --------------------------------------------------------------------------
# Prompt -> metin
# --------------------------------------------------------------------------
def build_chat_text(tokenizer, report: str, few_shot: bool = True,
                    system_prompt: str = None) -> str:
    """Sohbet sablonunu uygulayip tek bir metin uret.

    `enable_thinking=False` neden gerekli (Faz 1.8 bulgusu):
    Qwen3 ailesi varsayilan olarak thinking mode'da calisir ve cevaptan once
    <think>...</think> blogu yazar. Bake-off'ta Qwen3-14B-AWQ'nun parse_ok'u
    0.000 cikti — 700 token'lik butce dusunmeye gitti, JSON'a hic sira gelmedi.
    Bu parametre eski sablonlarda yok, o yuzden TypeError yakalanip atlaniyor.
    """
    msgs = [{"role": "system", "content": system_prompt or P.SYSTEM_PROMPT}]
    if few_shot:
        msgs += P.build_few_shot_messages()
    msgs.append({"role": "user", "content": P.build_user_message(report)})
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True,
            enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)


# --------------------------------------------------------------------------
# Logprob -> guven
# --------------------------------------------------------------------------
def status_confidences(output) -> dict:
    """Uretilen token'lar arasinda status kelimelerini bul, logprob'dan guven turet.

    Dondurur: {"present": [olasiliklar], "absent": [...], "uncertain": [...]}
    Sira, JSON'da etiketlerin gecis sirasiyla ayni (sema 'required' sirasini korur),
    boylece i. status token'i i. etikete denk gelir.

    Neden basit bir tarama: gudumlu decoding JSON yapisini garanti ettigi icin
    status alanlarinin sirasi deterministik. Token'i tam konumlandirmak yerine
    "status kelimesine denk gelen token"i yakalamak yeterli ve surume dayanikli.
    """
    found = {w: [] for w in STATUS_WORDS}
    lps = getattr(output, "logprobs", None)
    if not lps:
        return found
    for step in lps:
        if not step:
            continue
        # Bu adimda secilen token (en yuksek logprob'a sahip olan degil, SECILEN)
        try:
            chosen = max(step.values(), key=lambda x: getattr(x, "rank", 99) * -1
                         if hasattr(x, "rank") else x.logprob)
        except Exception:
            continue
        tok = (getattr(chosen, "decoded_token", None) or "").strip().strip('"').lower()
        if tok in STATUS_WORDS:
            found[tok].append(float(math.exp(chosen.logprob)))
    return found


def attach_logprob_conf(rec: dict, output) -> dict:
    """parse_response ciktisina logprob tabanli guveni ekle (<etiket>_lpconf).

    TAMAMI korumali: logprob cikti formati vLLM surumleri arasinda degisiyor
    (0.29'da `flat_logprobs` / `logprob_token_ids` gibi yeni alanlar var). Bir
    format degisikliginin 13 saatlik kosuyu oldurmesine izin veremeyiz —
    cozulemezse `_lpconf` None kalir, baska hicbir sey etkilenmez.

    Zaten OLCULDU (Faz 1.8): modelin kendi bildirdigi guven (AUC 0.819)
    logprob'dan turetilenden (0.679) daha iyi. Yani bu alan artik yedek bilgi.
    """
    try:
        conf = status_confidences(output)
        queues = {w: list(v) for w, v in conf.items()}
    except Exception:
        queues = {}
    for k in P.LABELS:
        try:
            q = queues.get(rec.get(f"{k}_status"))
            rec[f"{k}_lpconf"] = float(q.pop(0)) if q else None
        except Exception:
            rec[f"{k}_lpconf"] = None
    return rec


# --------------------------------------------------------------------------
# Ana kosu
# --------------------------------------------------------------------------
def run(model_id, reports, out_path, tensor_parallel_size=1,
        max_model_len=4096, gpu_memory_utilization=0.90, quantization=None,
        few_shot=True, max_tokens=700, batch_log_every=200, dtype="auto",
        logprobs=None, system_prompt=None, tag=None):
    """reports: [(study_uid, report_text), ...]. Sonuclari out_path'e JSONL yazar.

    Zaten yazilmis study'leri atlar (resume). Dondurur: (kayitlar, meta).
    """
    from transformers import AutoTokenizer
    from vllm import LLM

    done = set()
    if os.path.exists(out_path):
        with open(out_path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    done.add(json.loads(line)["StudyInstanceUID"])
                except Exception:
                    pass
        print(f"  {len(done):,} rapor zaten islenmis, atlaniyor (resume)")

    todo = [(u, r) for u, r in reports if u not in done]
    if not todo:
        print("  yapilacak is yok")
        return [], {}

    print(f"Model yukleniyor: {model_id}")
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    # ONEK CACHE'I — en buyuk hiz kaldiraci (Faz 1.8 bulgusu).
    # Sistem prompt'u + few-shot ornekleri ~2.180 token ve 4.407 istegin
    # HEPSINDE birebir ayni. Bu acik degilken vLLM o oneki her rapor icin
    # sifirdan yeniden hesapliyor: ~9.6 milyon gereksiz prefill token'i.
    # Ayni kitabin ilk 50 sayfasini her bolumde bastan okumak gibi.
    llm_kwargs = dict(model=model_id, tensor_parallel_size=tensor_parallel_size,
                      max_model_len=max_model_len, dtype=dtype,
                      gpu_memory_utilization=gpu_memory_utilization,
                      enable_prefix_caching=True,
                      trust_remote_code=True)
    if quantization:
        llm_kwargs["quantization"] = quantization
    try:
        llm = LLM(**llm_kwargs)
    except TypeError as e:
        # Eski vLLM surumlerinde parametre adi farkli olabilir
        print(f"  [bilgi] enable_prefix_caching kabul edilmedi ({e}); onek")
        print("          cache'i olmadan devam ediliyor — kosu YAVAS olacak.")
        llm_kwargs.pop("enable_prefix_caching")
        llm = LLM(**llm_kwargs)
    load_s = time.time() - t0
    print(f"  yuklendi ({load_s:.0f}s)")

    # logprobs neden varsayilan KAPALI: her token icin 5 aday dondurmek
    # 4.407 rapor x ~500 token x 5 = ~11M kayit serilestirmek demek ve olcumu
    # zaten yaptik — modelin kendi bildirdigi guven (AUC 0.819) logprob'dan
    # turetilenden (0.679) iyi cikti. Yeniden olcmek isteyen logprobs=5 verir.
    sp, guided_mode = make_sampling_params(P.output_schema(), max_tokens=max_tokens,
                                          logprobs=logprobs)
    print(f"  gudumlu decoding: {guided_mode}")
    print(f"  logprobs: {logprobs if logprobs else 'kapali (hiz icin)'}")

    prompts = [build_chat_text(tok, r, few_shot, system_prompt) for _, r in todo]

    # BUTCE KONTROLU — bu kontrol olmadigi icin Tur 3'te sessizce veri kaybettik.
    # vLLM girdi+cikti <= max_model_len olacak sekilde cikti butcesini KIRPAR ve
    # uyari vermez; JSON yarim kalir, ayristirma basarisiz olur, sebebi de
    # gorunmez. En uzun promptu olcup baştan haber veriyoruz.
    lens = [len(tok(pr).input_ids) for pr in prompts]
    n_in, n_max = sum(lens) / len(lens), max(lens)
    bos = max_model_len - n_max
    print(f"  girdi uzunlugu: ortalama {n_in:.0f}, EN UZUN {n_max} token")
    print(f"  max_model_len={max_model_len} -> en uzun promptta cikti icin"
          f" kalan: {bos} token  (istenen: {max_tokens})")
    if bos < max_tokens:
        kritik = sum(1 for L in lens if max_model_len - L < max_tokens)
        print("  " + "!" * 66)
        print(f"  !! BUTCE YETERSIZ: {kritik}/{len(lens)} promptta cikti butcesi")
        print(f"  !! {max_tokens} token'in ALTINA dusuyor. Bu raporlarda JSON")
        print(f"  !! yarim kalacak ve ayristirilamayacak.")
        print(f"  !! COZUM: max_model_len >= {n_max + max_tokens} yap.")
        print("  " + "!" * 66)
    else:
        print("  >> butce yeterli: tum promptlar tam cikti alabilir.")

    t0 = time.time()
    outs = llm.generate(prompts, sp)
    gen_s = time.time() - t0

    recs = []
    n_out = 0
    with open(out_path, "a", encoding="utf-8") as fh:
        for (uid, _), o in zip(todo, outs):
            gen = o.outputs[0]
            n_out += len(gen.token_ids)
            rec = P.parse_response(gen.text, uid)
            rec = attach_logprob_conf(rec, gen)
            rec["n_out_tokens"] = len(gen.token_ids)
            rec["model"] = model_id
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            recs.append(rec)

    ok = sum(r["parse_ok"] for r in recs)
    # Kesilme teshisi: gudumlu decoding modeli 12 bulgunun HEPSINI doldurmaya
    # zorluyor, kisa yoldan cikamiyor. Butce biterse JSON yarim kalir ve
    # ayristirilamaz. Tur 3'te parse_ok gudumlu decoding ACIKKEN dustu — bu
    # sayac o hipotezi dogrudan test ediyor.
    n_capped = sum(1 for r in recs if r.get("n_out_tokens", 0) >= max_tokens)
    n_bad_capped = sum(1 for r in recs
                       if not r["parse_ok"] and r.get("n_out_tokens", 0) >= max_tokens)
    meta = {
        "model": model_id, "tag": tag or model_id, "n": len(recs),
        "parse_ok": ok,
        "parse_ok_rate": ok / max(len(recs), 1),
        "guided_mode": guided_mode, "load_s": round(load_s, 1),
        "gen_s": round(gen_s, 1), "out_tokens": n_out,
        "tok_per_s": round(n_out / max(gen_s, 1e-9), 1),
        "s_per_report": round(gen_s / max(len(recs), 1), 3),
        "mean_out_tokens": round(n_out / max(len(recs), 1), 1),
        "max_tokens": max_tokens,
        "max_model_len": max_model_len,
        "max_prompt_tokens": int(n_max),
        "budget_ok": bool(max_model_len - n_max >= max_tokens),
        "n_capped": n_capped,
        "n_fail_capped": n_bad_capped,
        "n_fail": len(recs) - ok,
    }
    # Teshis DOSYAYA da yazilir: vLLM binlerce INFO satiri basiyor ve Kaggle
    # cikti limitini asinca kirpiyor — ilk kosuda "gudumlu decoding" satiri tam
    # bu yuzden kayboldu ve gudumsuz calistigini fark etmedik.
    try:
        diag = os.path.join(os.path.dirname(out_path) or ".", "run_diagnostics.jsonl")
        with open(diag, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(meta, ensure_ascii=False) + "\n")
    except Exception:
        pass
    print(f"  {len(recs):,} rapor / {gen_s:.0f}s  "
          f"({meta['s_per_report']:.2f} s/rapor, {meta['tok_per_s']:.0f} tok/s)")
    print(f"  ayristirma basarisi: {ok}/{len(recs)} ({meta['parse_ok_rate']:.1%})")
    print(f"  token tavanina dayanan: {n_capped}  |  bunlardan ayristirilamayan:"
          f" {n_bad_capped} / {len(recs)-ok}")
    est_h = meta["s_per_report"] * 4407 / 3600
    print(f"  >> 4.407 raporun tamami icin tahmin: {est_h:.2f} saat")
    return recs, meta

## 3. Ortam kontrolü

GPU ve VRAM'i tespit ediyoruz — hangi adayların sığdığını bu belirliyor.

In [ ]:
import json
import os
import subprocess
import sys
import time

sys.path.insert(0, "/kaggle/working")

import numpy as np
import pandas as pd
import torch

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print("torch      :", torch.__version__)
print("CUDA var mi:", torch.cuda.is_available())
N_GPU = torch.cuda.device_count()
print("GPU sayisi :", N_GPU)
VRAM_GB = 0.0
VRAM_PER_GPU = 0.0
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    gb = p.total_memory / 1024**3
    VRAM_GB += gb
    VRAM_PER_GPU = gb if i == 0 else min(VRAM_PER_GPU, gb)
    print(f"  GPU {i}: {p.name}  {gb:.1f} GB")
print(f"TOPLAM VRAM : {VRAM_GB:.1f} GB")
print(f"GPU BASINA  : {VRAM_PER_GPU:.1f} GB   <- bir model ancak tp x bu kadarini kullanir")

if N_GPU == 0:
    raise RuntimeError("GPU yok. Sag panel -> Accelerator -> GPU secin.")

## 4. Veri: dev set

UID listeleri koda gömülü (`devset_ids.py`), ayrı bir Kaggle Dataset gerekmiyor.

In [ ]:
import prompt as P
import devset_ids as D
import evaluate as EV

def find_train_csv(root="/kaggle/input", depth=2):
    """train.csv'yi iki seviyeye kadar ara. Bulamazsa NE MOUNT EDILMIS oldugunu yaz.

    Onceki surum sadece tek seviye ariyordu ve bulamayinca sadece "bulunamadi"
    diyordu — hangi dataset'lerin bagli oldugunu gostermek teshisi cok kolaylastirir.
    """
    if not os.path.isdir(root):
        return None, []
    seen = []
    for a in sorted(os.listdir(root)):
        pa = os.path.join(root, a)
        if not os.path.isdir(pa):
            continue
        seen.append(a)
        if os.path.exists(os.path.join(pa, "train.csv")):
            return pa, seen
        if depth > 1:
            for b in sorted(os.listdir(pa)):
                pb = os.path.join(pa, b)
                if os.path.isdir(pb) and os.path.exists(os.path.join(pb, "train.csv")):
                    return pb, seen
    return None, seen


DATA, mounted = find_train_csv()
if DATA is None:
    print("Mount edilmis dataset'ler:", mounted if mounted else "(HICBIRI)")
    print()
    print("  Sag panel -> '+ Add Input' -> Competitions sekmesi ->")
    print("  'rsna-knee-abnormality-detection' -> Add")
    print()
    print("  NOT: Import edilen notebook input'lari BERABERINDE GETIRMEZ.")
    print("  Her import sonrasi veriyi yeniden baglamak gerekiyor.")
    raise FileNotFoundError("train.csv bulunamadi — yukaridaki adimlari izleyin.")
print("VERI:", DATA)

train = pd.read_csv(f"{DATA}/train.csv")
train["Report"] = train["Report"].fillna("")
by_uid = train.set_index("StudyInstanceUID")

# HATA DUZELTMESI: eskiden `if MODE == "bakeoff"` ... `else: tum veri` yaziyordu.
# "ablation" modu else dalina dustu ve 49 rapor yerine 4.407 raporu isledi —
# 8.4 saatlik kazara bir tam kosu. (Sonuc iyi cikti ama niyet bu degildi.)
# Artik mod ADI ile eslestiriliyor, bilinmeyen mod hata veriyor.
if MODE == "retry":
    # Tam kosuda 1100 token tavanina dayanip kesilen 53 rapor. Sadece bunlari
    # yeniden islemek 4.407'yi bastan almaktan ~80 kat hizli.
    import retry_ids as R
    uids = [u for u in R.RETRY if u in by_uid.index]
elif MODE in ("bakeoff", "ablation"):
    uids = [u for u in D.DEVSET if u in by_uid.index]
elif MODE == "full":
    uids = train["StudyInstanceUID"].tolist()
else:
    raise ValueError(f"bilinmeyen MODE: {MODE!r}. "
                     'Gecerli: "retry" | "ablation" | "bakeoff" | "full"')
_ne = {"retry": "kesilen 53 rapor", "bakeoff": "49 raporluk dev set",
       "ablation": "49 raporluk dev set", "full": "TUM train seti"}
print(f"  (mod '{MODE}' -> {_ne[MODE]})")
reports = [(u, by_uid.loc[u, "Report"]) for u in uids]
print(f"MODE={MODE}  ->  {len(reports):,} rapor")

# Yer gercegi: dev set'in gold olanlari (bake-off degerlendirmesi bunlarda)
gold_cols = ["StudyInstanceUID"] + P.LABELS
gold_dev = train[train["StudyInstanceUID"].isin(D.GOLD_DEV)][gold_cols].copy()
print(f"gold_dev (olcum icin): {len(gold_dev)} study")
print(f"gold_holdout (DOKUNULMAYACAK): {len(D.GOLD_HOLDOUT)} study")

## 5. Ön uçuş kontrolü — model var mı, sığıyor mu?

İki eleme birden, **indirmeden önce**:

1. **Model gerçekten var mı?** Hugging Face'te depo adları değişiyor ve bazı
   kuvantize varyantlar topluluk tarafından üretiliyor, resmi olmayabilir.
   20 GB indirdikten sonra 404 almak istemiyoruz — `model_info` çağrısı
   saniyeler sürer ve hiçbir şey indirmez.
2. **VRAM'e sığıyor mu?** `vram_gb` tahminleri kaba; sığmayan model zaten
   hata verip atlanır ama baştan elemek oturum süresi kazandırır.

Yerel bir yol (`/kaggle/input/...`) verilmişse HF kontrolü atlanır.

In [ ]:
def model_exists(model_id):
    """(var_mi, aciklama). Hicbir sey indirmez."""
    if model_id.startswith("/"):
        return os.path.exists(model_id), "yerel yol"
    try:
        from huggingface_hub import model_info
        info = model_info(model_id)
        n = len(getattr(info, "siblings", None) or [])
        return True, f"HF'de var ({n} dosya)"
    except Exception as e:
        return False, f"{type(e).__name__}"


print("ON UCUS KONTROLU")
print("-" * 72)
fits = []
for c in CANDIDATES:
    exists, note = model_exists(c["id"])
    # DUZELTME (bake-off 1 bulgusu): bir model tp GPU'ya yayilir, yani
    # kullanabilecegi VRAM = tp x GPU_BASINA_VRAM. Onceki surum toplam VRAM ile
    # karsilastiriyordu; bu yuzden Qwen2.5-7B (fp16, ~15 GB) tp=1 ile "sigar"
    # gorunup tek 15 GB'lik T4'te OOM verdi. Karsilastirma yanlis taraftaydi.
    usable = VRAM_PER_GPU * c["tp"]
    if not exists:
        verdict, why = "ATLA", f"BULUNAMADI ({note})"
    elif c["tp"] > N_GPU:
        verdict, why = "ATLA", f"{c['tp']} GPU gerekiyor, {N_GPU} var"
    elif c["vram_gb"] > usable:
        verdict, why = "ATLA", (f"VRAM yetmez ({c['vram_gb']} > {usable:.0f} = "
                                f"{c['tp']}x{VRAM_PER_GPU:.0f})")
    else:
        verdict, why = "OK  ", f"{note}, sigar ({c['vram_gb']}/{usable:.0f} GB)"
    print(f"  [{verdict}] {c['id']:<40} {why}")
    if verdict.strip() == "OK":
        fits.append(c)

print("-" * 72)
print(f"Yarisacak model: {len(fits)} / {len(CANDIDATES)}")
if not fits:
    raise RuntimeError(
        "Hicbir aday kullanilabilir degil.\n"
        "  - 'BULUNAMADI' ise: HF'de dogru depo adini bulup CANDIDATES'i guncelle\n"
        "    (ornek arama: huggingface.co/models?search=qwen2.5+instruct+awq)\n"
        "  - 'VRAM yetmez' ise: daha kucuk veya kuvantize bir model ekle\n"
        "  - Kaggle Models sekmesinden model ekleyip 'id' yerine\n"
        "    '/kaggle/input/<dataset>/<yol>' yazmak indirme suresini sifirlar")

## 6. transformers yedeği

vLLM yoksa bu kullanılır: güdümlü decoding yok, sadece batched `generate`.
Ayrıştırıcımız dayanıklı olduğu için geçersiz JSON tamamen kaybedilmiyor,
`missing` olarak işaretleniyor — ama ayrıştırma başarısı oranını izle.

In [ ]:
def run_transformers(model_id, reports, out_path, quant=None, batch_size=8):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    done = set()
    if os.path.exists(out_path):
        with open(out_path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    done.add(json.loads(line)["StudyInstanceUID"])
                except Exception:
                    pass
    todo = [(u, r) for u, r in reports if u not in done]
    if not todo:
        return [], {}

    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True,
                                        padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto",
        trust_remote_code=True)
    model.eval()
    load_s = time.time() - t0

    recs, n_out = [], 0
    t0 = time.time()
    with open(out_path, "a", encoding="utf-8") as fh:
        for i in range(0, len(todo), batch_size):
            chunk = todo[i:i + batch_size]
            texts = []
            for _, r in chunk:
                msgs = ([{"role": "system", "content": P.SYSTEM_PROMPT}]
                        + P.build_few_shot_messages()
                        + [{"role": "user", "content": P.build_user_message(r)}])
                texts.append(tok.apply_chat_template(msgs, tokenize=False,
                                                     add_generation_prompt=True))
            enc = tok(texts, return_tensors="pt", padding=True, truncation=True,
                      max_length=MAX_MODEL_LEN).to(model.device)
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                                     do_sample=False,
                                     pad_token_id=tok.pad_token_id)
            for (uid, _), row in zip(chunk, gen):
                new = row[enc["input_ids"].shape[1]:]
                n_out += int((new != tok.pad_token_id).sum())
                rec = P.parse_response(tok.decode(new, skip_special_tokens=True), uid)
                rec["model"] = model_id
                for k in P.LABELS:
                    rec[f"{k}_lpconf"] = None      # bu yolda logprob toplanmiyor
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
                recs.append(rec)
            print(f"    {min(i+batch_size, len(todo))}/{len(todo)}", end="\r")
    gen_s = time.time() - t0
    del model
    torch.cuda.empty_cache()

    ok = sum(r["parse_ok"] for r in recs)
    return recs, {"model": model_id, "n": len(recs), "parse_ok": ok,
                  "parse_ok_rate": ok / max(len(recs), 1),
                  "mean_out_tokens": round(n_out / max(len(recs), 1), 1),
                  "guided_mode": "yok (transformers)", "load_s": round(load_s, 1),
                  "gen_s": round(gen_s, 1), "out_tokens": int(n_out),
                  "tok_per_s": round(n_out / max(gen_s, 1e-9), 1),
                  "s_per_report": round(gen_s / max(len(recs), 1), 3)}

## 6b. Ablation — tek değişken: "minimal → uncertain" kuralı

Tur 3'te dört şey birden değişti (güdümlü decoding, önek cache'i, prompt
kuralları, logprobs) ve `F1_absent` 0.7251 → 0.6805 geriledi. Hangisinin
yaptığını bilmiyoruz. Bu bölüm **sadece 7. kuralı** açıp kapatıp ölçüyor;
diğer her şey sabit.

Aynı zamanda `MAX_NEW_TOKENS = 1100` ile kesilme hipotezini test ediyor:
`n_capped` sayacı sıfıra yakınsa `parse_ok` düşüşünün sebebi kesilmeydi.

In [ ]:
if MODE == "ablation":
    import vllm_runner as VR

    abl_rows, abl_ev = [], {}
    for vtag, with_rule in ABLATION_VARIANTS:
        print("=" * 72)
        print(f"VARYANT: {vtag}   (7. kural {'VAR' if with_rule else 'YOK'})")
        print("=" * 72)
        sys_prompt = P.build_system_prompt(with_minimal_rule=with_rule)
        print(f"  sistem prompt: {len(sys_prompt)} karakter")
        out_path = f"{OUT_DIR}/abl_{vtag}.jsonl"
        try:
            recs, meta = VR.run(
                ABLATION_MODEL["id"], reports, out_path,
                tensor_parallel_size=ABLATION_MODEL["tp"],
                max_model_len=MAX_MODEL_LEN, quantization=ABLATION_MODEL["quant"],
                max_tokens=MAX_NEW_TOKENS, logprobs=LOGPROBS,
                system_prompt=sys_prompt, tag=vtag)
        except Exception as e:
            print(f"  BASARISIZ: {type(e).__name__}: {str(e)[:300]}")
            abl_rows.append({"varyant": vtag, "durum": f"HATA: {type(e).__name__}"})
            continue
        if not recs:
            recs = [json.loads(l) for l in open(out_path, encoding="utf-8")]
        pred = pd.DataFrame(recs)

        row = {"varyant": vtag, "durum": "ok",
               "parse_ok": round(meta.get("parse_ok_rate", np.nan), 3),
               "tavana_dayanan": meta.get("n_capped"),
               "cikti_token": meta.get("mean_out_tokens"),
               "s_rapor": meta.get("s_per_report"),
               "tam_kosu_saat": round(meta.get("s_per_report", 0) * 4407 / 3600, 2)}
        for unc in ("mask", "absent", "half"):
            ev = EV.evaluate(pred, gold_dev, uncertain=unc)
            s = EV.summarise(ev)
            row[f"F1_{unc}"] = s["macro_F1"]
            row[f"AUC_{unc}"] = s["macro_AUC"]
            if unc == "mask":
                row["kapsam"] = s["ort_kapsam"]
                abl_ev[vtag] = ev
        # Guven kalitesi de varyanta gore degisebilir
        cq = EV.confidence_quality(pred, gold_dev, conf_suffix="_conf")
        row["AUC_guven"] = (round(float(cq["AUC_guven"].mean(skipna=True)), 3)
                            if len(cq) and cq["AUC_guven"].notna().any() else None)
        abl_rows.append(row)
        print()

    abl = pd.DataFrame(abl_rows)
    print("=" * 72)
    print("ABLATION SONUCLARI")
    print("=" * 72)
    print(abl.to_string(index=False))
    abl.to_csv(f"{OUT_DIR}/ablation_summary.csv", index=False)

    ok_rows = abl[abl["durum"] == "ok"] if "durum" in abl.columns else abl
    if len(ok_rows) == 2:
        a = ok_rows.iloc[0]
        b = ok_rows.iloc[1]
        print()
        print("--- KARAR ---")
        for k in ("F1_absent", "AUC_half", "kapsam", "parse_ok"):
            print(f"  {k:<12} {a['varyant']}={a[k]}   {b['varyant']}={b[k]}"
                  f"   fark={b[k]-a[k]:+.4f}")
        kazanan = a if a["F1_absent"] >= b["F1_absent"] else b
        print()
        print(f"  >> F1_absent'e gore kazanan: {kazanan['varyant']}")
        print(f"  >> Tur 1 (eski prompt, gudumsuz) referansi: F1_absent 0.7251")

## 7. Bake-off

Her model için: çalıştır → gold_dev'de değerlendir → tabloya ekle.
Sonuçlar JSONL'e anında yazıldığı için oturum kesilirse kaldığı yerden devam eder.

In [ ]:
if MODE == "bakeoff":
    import vllm_runner as VR

    results, all_ev = [], {}
    for c in fits:
        tag = c["id"].split("/")[-1]
        print("=" * 72)
        print(c["id"])
        print("=" * 72)
        out_path = f"{OUT_DIR}/raw_{tag}.jsonl"
        try:
            if HAS_VLLM:
                recs, meta = VR.run(
                    c["id"], reports, out_path,
                    tensor_parallel_size=c["tp"], max_model_len=MAX_MODEL_LEN,
                    quantization=c["quant"], max_tokens=MAX_NEW_TOKENS,
                    logprobs=LOGPROBS)
            else:
                recs, meta = run_transformers(c["id"], reports, out_path, c["quant"])
        except Exception as e:
            print(f"  BASARISIZ: {type(e).__name__}: {str(e)[:300]}")
            results.append({"model": tag, "durum": f"HATA: {type(e).__name__}"})
            continue

        if not recs:
            recs = [json.loads(l) for l in open(out_path, encoding="utf-8")]
        pred = pd.DataFrame(recs)

        # guided_mode TABLOYA girsin: ilk kosuda bu bilgi sadece ekrana basiliyordu
        # ve vLLM log seli icinde kayboldu — gudumsuz calistigini fark etmedik.
        row = {"model": tag, "durum": "ok",
               "gudumlu": meta.get("guided_mode", "?"),
               "parse_ok": round(meta.get("parse_ok_rate", np.nan), 3),
               "cikti_token": meta.get("mean_out_tokens"),
               "s_rapor": meta.get("s_per_report"),
               "tam_kosu_saat": round(meta.get("s_per_report", 0) * 4407 / 3600, 2)}
        for unc in ("mask", "absent", "half"):
            ev = EV.evaluate(pred, gold_dev, uncertain=unc)
            s = EV.summarise(ev)
            row[f"F1_{unc}"] = s["macro_F1"]
            row[f"AUC_{unc}"] = s["macro_AUC"]
            if unc == "mask":
                row["kapsam"] = s["ort_kapsam"]
                row["en_zayif"] = s["en_zayif"]
                all_ev[tag] = ev
        results.append(row)
        print()

    bake = pd.DataFrame(results)
    print("=" * 72)
    print("BAKE-OFF SONUCLARI")
    print("=" * 72)
    print(bake.to_string(index=False))
    bake.to_csv(f"{OUT_DIR}/bakeoff_summary.csv", index=False)

    # Bu kontrol ekrana ayrica basiliyor cunku log seli icinde kaybolmamasi lazim.
    if "gudumlu" in bake.columns:
        bad = bake[bake["gudumlu"].astype(str).str.contains("GUDUMSUZ|yok", na=False)]
        print()
        if len(bad):
            print("!" * 72)
            print("!! GUDUMLU DECODING CALISMADI:", list(bad["model"]))
            print("!! Model semaya uymak zorunda DEGIL -> parse_ok dusuk kalir.")
            print("!! Bu satiri bana gonder, vLLM surumune gore duzeltelim.")
            print("!" * 72)
        else:
            print(">> Gudumlu decoding TUM modellerde aktif:",
                  list(bake["gudumlu"].unique()))

### 7b. Kazananın etiket bazında dökümü

`macro_F1` tek bir sayı; hangi etiketin zayıf olduğunu görmek Katman 4'teki
ağırlıklandırmayı belirleyecek.

In [ ]:
if MODE == "bakeoff" and len(all_ev):
    ok = bake[bake["durum"] == "ok"]
    if len(ok):
        best = ok.loc[ok["F1_absent"].idxmax(), "model"]
        print(f"En iyi (F1_absent'e gore): {best}")
        print()
        print(all_ev[best].to_string(index=False))
        all_ev[best].to_csv(f"{OUT_DIR}/bakeoff_best_per_label.csv", index=False)

        # Guven skoru: modelin kendi bildirdigi mi, logprob mu daha iyi?
        recs = [json.loads(l) for l in
                open(f"{OUT_DIR}/raw_{best}.jsonl", encoding="utf-8")]
        pred = pd.DataFrame(recs)
        print()
        print("--- GUVEN SKORU KALITESI ---")
        for suf, isim in [("_conf", "modelin kendi bildirdigi"),
                          ("_lpconf", "logprob'dan turetilen")]:
            cq = EV.confidence_quality(pred, gold_dev, conf_suffix=suf)
            if len(cq) and cq["AUC_guven"].notna().any():
                m = cq["AUC_guven"].mean(skipna=True)
                print(f"  {isim:<26} ortalama AUC_guven = {m:.3f}")
            else:
                print(f"  {isim:<26} olculemedi (sabit veya eksik)")
        print()
        print("  AUC_guven 0.5 = guven rastgele, dogrulukla ilgisi yok.")
        print("  0.5'e yakinsa Katman 4'te guvene gore agirliklandirma ZARAR verir;")
        print("  o durumda duz agirlik kullanilir.")

## 7c. Retry — kesilen raporları yeniden işle

Tam koşuda 53 rapor (%1.2) `max_tokens` tavanına dayanıp kesildi; JSON yarım
kaldı. Hepsinin sebebi aynıydı (`n_fail_capped == n_fail == 53`), yani
`MAX_NEW_TOKENS = 1400` ile düzelmesi bekleniyor.

Sadece o 53'ü işliyoruz — 4.407'yi baştan almak ~8.4 saat, bu ~7 dakika.

In [ ]:
if MODE == "retry":
    import vllm_runner as VR

    tag = FULL_RUN_MODEL["id"].split("/")[-1] if FULL_RUN_MODEL else "retry"
    out_path = f"{OUT_DIR}/retry_{tag}.jsonl"
    recs, meta = VR.run(FULL_RUN_MODEL["id"], reports, out_path,
                        tensor_parallel_size=FULL_RUN_MODEL["tp"],
                        max_model_len=MAX_MODEL_LEN,
                        quantization=FULL_RUN_MODEL["quant"],
                        max_tokens=MAX_NEW_TOKENS, logprobs=LOGPROBS, tag="retry")
    print(json.dumps(meta, ensure_ascii=False, indent=2))

    df_r = pd.DataFrame(recs if recs else
                        [json.loads(l) for l in open(out_path, encoding="utf-8")])
    print()
    print(f"Yeniden islenen : {len(df_r)}")
    print(f"Basarili        : {int(df_r.parse_ok.sum())} / {len(df_r)}"
          f"  ({df_r.parse_ok.mean():.1%})")
    print(f"Hala tavanda    : {int((df_r.n_out_tokens >= MAX_NEW_TOKENS).sum())}")
    if df_r.parse_ok.mean() == 1.0:
        print("  >> HEPSI KURTARILDI. Faz 1.8 tam kapsam: 4407/4407")
    else:
        kalan = int((~df_r.parse_ok).sum())
        print(f"  >> {kalan} rapor hala basarisiz. Tavanda iseler MAX_NEW_TOKENS")
        print("     daha da artirilabilir; degilse baska bir sebep var.")

## 8. Tam koşu

`MODE = "full"` ve `FULL_RUN_MODEL` doldurulduğunda çalışır.
JSONL'e anında yazdığı için oturum kesilirse aynı notebook'u tekrar
çalıştırmak kaldığı yerden devam ettirir.

In [ ]:
if MODE == "full":
    if FULL_RUN_MODEL is None:
        raise ValueError("FULL_RUN_MODEL'i bake-off kazananiyla doldurun.")
    if not HAS_VLLM:
        print("!! UYARI: vLLM yok. 4.407 rapor transformers ile cok uzun surer.")
        print("   Yine de devam ediliyor — JSONL resume oldugu icin birden fazla")
        print("   oturumda tamamlanabilir.")
    import vllm_runner as VR

    tag = FULL_RUN_MODEL["id"].split("/")[-1]
    out_path = f"{OUT_DIR}/weak_raw_{tag}.jsonl"
    if HAS_VLLM:
        recs, meta = VR.run(FULL_RUN_MODEL["id"], reports, out_path,
                            tensor_parallel_size=FULL_RUN_MODEL["tp"],
                            max_model_len=MAX_MODEL_LEN,
                            quantization=FULL_RUN_MODEL["quant"],
                            max_tokens=MAX_NEW_TOKENS, logprobs=LOGPROBS)
    else:
        recs, meta = run_transformers(FULL_RUN_MODEL["id"], reports, out_path,
                                      FULL_RUN_MODEL["quant"])
    print(json.dumps(meta, ensure_ascii=False, indent=2))

    allr = [json.loads(l) for l in open(out_path, encoding="utf-8")]
    df = pd.DataFrame(allr)
    print(f"\nToplam islenmis: {len(df):,} / {len(train):,}")
    print(f"Ayristirma basarisi: {df['parse_ok'].mean():.1%}")
    print("\nStatus dagilimi (tum etiketler birlestirilmis):")
    st = pd.concat([df[f"{k}_status"] for k in P.LABELS])
    print((st.value_counts(normalize=True) * 100).round(1).to_string())

    keep = (["StudyInstanceUID", "parse_ok", "exam_completeness", "model"]
            + [c for k in P.LABELS for c in
               (k, f"{k}_status", f"{k}_conf", f"{k}_lpconf", f"{k}_evidence")])
    df[[c for c in keep if c in df.columns]].to_csv(
        f"{OUT_DIR}/weak_labels_layer1.csv", index=False)
    print(f"\nkaydedildi: {OUT_DIR}/weak_labels_layer1.csv")

    # GOLD HOLDOUT SAGLAMASI — bu 38 study prompt ayarinda hic kullanilmadi
    gh = train[train["StudyInstanceUID"].isin(D.GOLD_HOLDOUT)][gold_cols]
    ev = EV.evaluate(df, gh, uncertain="absent")
    print()
    print("=" * 72)
    print("KATMAN 3 — GOLD HOLDOUT UZERINDE NIHAI OLCUM (38 study)")
    print("=" * 72)
    print(ev.to_string(index=False))
    print()
    print(json.dumps(EV.summarise(ev), ensure_ascii=False, indent=2))
    ev.to_csv(f"{OUT_DIR}/layer3_holdout_eval.csv", index=False)
    print()
    print("  Bu tablo Faz 2.8'deki etiket basina loss agirliklandirmasini belirler:")
    print("  F1'i dusuk etiketlerde weak-label agirligi dusurulur.")

## 9. TESLİM ÖZETİ — sadece bu hücrenin çıktısını göndermek yeterli

**Neden en sonda ve ayrı bir hücre:** vLLM binlerce `INFO` satırı basıyor ve
Kaggle uzun çıktıları kırpıyor. Üç turdur kritik satırlar (`gudumlu decoding`,
özet tablo) o selin içinde kayboldu. Bu hücre en son çalıştığı için önündeki
log seli onu etkilemiyor — çıktısı kısa ve eksiksiz.

**Çıktıların kalıcı olması için `Save Version → Save & Run All` kullan.**
Interaktif oturumda `/kaggle/working` oturum bitince uçuyor (No persistence).

In [ ]:
print("=" * 72)
print("TESLIM OZETI")
print("=" * 72)

print()
print("--- /kaggle/working icerigi ---")
for f in sorted(os.listdir(OUT_DIR)):
    try:
        kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
        print(f"  {kb:9.1f} KB  {f}")
    except OSError:
        print(f"  {'?':>9}     {f}")

for ad, yol in [("ABLATION SUMMARY", f"{OUT_DIR}/ablation_summary.csv"),
                ("BAKEOFF SUMMARY", f"{OUT_DIR}/bakeoff_summary.csv"),
                ("KAZANAN — ETIKET BAZINDA", f"{OUT_DIR}/bakeoff_best_per_label.csv"),
                ("KATMAN 3 — GOLD HOLDOUT", f"{OUT_DIR}/layer3_holdout_eval.csv")]:
    print()
    print(f"--- {ad} ---")
    if os.path.exists(yol):
        print(pd.read_csv(yol).to_string(index=False))
    else:
        print("  (yok — bu mod calismadi)")

print()
print("--- RUN DIAGNOSTICS (model basina altyapi) ---")
dp = f"{OUT_DIR}/run_diagnostics.jsonl"
if os.path.exists(dp):
    for line in open(dp, encoding="utf-8"):
        d = json.loads(line)
        print(f"  {d.get('tag') or d.get('model','?')}")
        print(f"      gudumlu={d.get('guided_mode')}")
        print(f"      butce_ok={d.get('budget_ok')}"
              f"  max_model_len={d.get('max_model_len')}"
              f"  en_uzun_prompt={d.get('max_prompt_tokens')}"
              f"  max_tokens={d.get('max_tokens')}")
        print(f"      tavana_dayanan={d.get('n_capped')}/{d.get('n')}"
              f"  (bunlardan ayristirilamayan: {d.get('n_fail_capped')}"
              f" / toplam hata {d.get('n_fail')})")
        print(f"      parse_ok={d.get('parse_ok')}/{d.get('n')}"
              f"  cikti_token_ort={d.get('mean_out_tokens')}"
              f"  {d.get('s_per_report')} s/rapor"
              f"  -> 4407 rapor ~{round(d.get('s_per_report',0)*4407/3600,2)} saat")
else:
    print("  (yok)")

# Tam kosu yapildiysa etiket dagilimini da ozetle
wl = f"{OUT_DIR}/weak_labels_layer1.csv"
if os.path.exists(wl):
    w = pd.read_csv(wl)
    print()
    print("--- TAM KOSU DURUMU ---")
    print(f"  islenmis study: {len(w):,} / 4407   ayristirma: {w['parse_ok'].mean():.1%}")
    st = pd.concat([w[f"{k}_status"] for k in P.LABELS if f"{k}_status" in w.columns])
    print("  status dagilimi: " +
          "  ".join(f"{k}={v:.1%}" for k, v in st.value_counts(normalize=True).items()))

print()
print("=" * 72)
print("Bu blogu oldugu gibi gonder. Ayrica Output sekmesinden indir:")
print("  bakeoff_summary.csv  ·  bakeoff_best_per_label.csv  ·  run_diagnostics.jsonl")
print("=" * 72)